# Building Transparent Loan Approval Prediction System

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('LoanPredictionProblemDataset.csv')
print(df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print(df.info())
print('\nMissing Values:\n', df.isnull().sum())
print('\nDescriptive Stats:\n')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Loan_Status'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon'])
axes[0].set_title('Loan Status Distribution')
axes[0].set_xlabel('Loan Status')

df['ApplicantIncome'].plot(kind='hist', bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Applicant Income Distribution')

plt.tight_layout()
plt.show()

cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']
fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 4))
for ax, col in zip(axes, cat_cols):
    df[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 3. Data Cleaning

### 4.1 Handle Missing Values

In [ ]:
df.drop(columns=['Loan_ID'], inplace=True)

cat_fill = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in cat_fill:
    df[col].fillna(df[col].mode()[0], inplace=True)

df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df = df.dropna(subset=['Loan_Status'])

print('Missing values after handling:\n', df.isnull().sum())

### Fix Incorrect Values

In [ ]:
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

df['ApplicantIncome'] = df['ApplicantIncome'].clip(lower=0)
df['CoapplicantIncome'] = df['CoapplicantIncome'].clip(lower=0)

print(df['Dependents'].value_counts())
print(df[['ApplicantIncome','CoapplicantIncome']].describe())

## 4. Encoding Categorical Data

In [ ]:
le = LabelEncoder()

binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Loan_Status']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)

print(df.dtypes)
df.head()

## 5. Feature Engineering

In [ ]:
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']
df['IncomeToLoan'] = df['TotalIncome'] / (df['LoanAmount'] + 1)

df.drop(columns=['ApplicantIncome', 'CoapplicantIncome'], inplace=True)

print(df[['TotalIncome','EMI','IncomeToLoan']].describe())

## 6. Outlier Removal using IQR

In [ ]:
num_cols = ['LoanAmount', 'TotalIncome', 'EMI', 'IncomeToLoan']

before = len(df)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

print(f'Rows before: {before}, after outlier removal: {len(df)}')

## 7. Feature Scaling — StandardScaler

In [ ]:
scale_cols = ['LoanAmount', 'Loan_Amount_Term', 'TotalIncome', 'EMI', 'IncomeToLoan']

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(df[scale_cols].describe().round(2))

## 8. Split Data

In [ ]:
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Train label distribution:\n{y_train.value_counts()}')

## 9. Train Random Forest Model

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

---
# Explainable AI and Model Interpretability

## 10. Feature Relevance — Filter Methods
Statistical tests to rank features independently of any model.

In [ ]:
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif

# Shift features to be non-negative for chi2
X_pos = X_train - X_train.min()

chi2_scores, _ = chi2(X_pos, y_train)
f_scores, _    = f_classif(X_train, y_train)
mi_scores      = mutual_info_classif(X_train, y_train, random_state=42)

filter_df = pd.DataFrame({
    'Feature'  : X_train.columns,
    'Chi2'     : chi2_scores,
    'F-Score'  : f_scores,
    'Mutual_Info': mi_scores
}).set_index('Feature').sort_values('Mutual_Info', ascending=False)

print(filter_df.round(4))

filter_df['Mutual_Info'].plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Filter Method — Mutual Information')
plt.xlabel('Score')
plt.tight_layout()
plt.show()

## 11. Feature Relevance — Wrapper Method (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr, n_features_to_select=6)
rfe.fit(X_train, y_train)

rfe_df = pd.DataFrame({
    'Feature' : X_train.columns,
    'Selected': rfe.support_,
    'Rank'    : rfe.ranking_
}).set_index('Feature').sort_values('Rank')

print(rfe_df)

colors = ['green' if s else 'salmon' for s in rfe_df['Selected']]
rfe_df['Rank'].plot(kind='barh', figsize=(8, 5), color=colors)
plt.title('Wrapper Method — RFE Ranking (lower = more important)')
plt.xlabel('Rank')
plt.tight_layout()
plt.show()

## 12. Feature Relevance — Embedded Method (Lasso)

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel

lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train, y_train)

lasso_coef = pd.Series(np.abs(lasso.coef_), index=X_train.columns).sort_values(ascending=False)
print('Lasso coefficients (absolute):')
print(lasso_coef.round(4))

lasso_coef.plot(kind='barh', figsize=(8, 5), color='darkorange')
plt.title('Embedded Method — Lasso |Coefficients|')
plt.xlabel('|Coefficient|')
plt.tight_layout()
plt.show()

## 13. Model-Specific Feature Importance (Random Forest)

In [ ]:
rf_importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print('Random Forest Feature Importances:')
print(rf_importance.round(4))

rf_importance.plot(kind='barh', figsize=(8, 5), color='teal')
plt.title('RF Model-Specific Feature Importance (MDI)')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

### 13.1 Permutation Importance (model-agnostic variant on test set)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

perm_df = pd.DataFrame({
    'Feature'       : X_test.columns,
    'Importance_Mean': perm.importances_mean,
    'Importance_Std' : perm.importances_std
}).set_index('Feature').sort_values('Importance_Mean', ascending=False)

print(perm_df.round(4))

perm_df['Importance_Mean'].plot(kind='barh', figsize=(8, 5),
                                 xerr=perm_df['Importance_Std'], color='purple')
plt.title('Permutation Importance on Test Set')
plt.xlabel('Mean Accuracy Decrease')
plt.tight_layout()
plt.show()

## 14. Model-Agnostic XAI — SHAP Analysis
Install if needed: `pip install shap`

In [ ]:
import shap

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification shap_values is a list [class0, class1]
# Use class-1 (loan approved)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print('SHAP values shape:', sv.shape)

### 14.1 SHAP Summary Plot (Global)

In [ ]:
shap.summary_plot(sv, X_test, plot_type='bar', show=False)
plt.title('SHAP — Global Feature Importance (Bar)')
plt.tight_layout()
plt.show()

shap.summary_plot(sv, X_test, show=False)
plt.title('SHAP — Beeswarm Summary Plot')
plt.tight_layout()
plt.show()

### 14.2 SHAP Local Explanation — Single Prediction (Waterfall)

In [ ]:
# Explain the first test instance
idx = 0
shap.plots.waterfall(
    shap.Explanation(
        values        = sv[idx],
        base_values   = explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                        else explainer.expected_value,
        data          = X_test.iloc[idx].values,
        feature_names = X_test.columns.tolist()
    ),
    show=False
)
plt.title(f'SHAP Waterfall — Test Sample {idx} (True label: {y_test.iloc[idx]})')
plt.tight_layout()
plt.show()

### 14.3 SHAP Dependence Plot — Top Feature

In [ ]:
top_feature = pd.Series(np.abs(sv).mean(axis=0), index=X_test.columns).idxmax()
print(f'Top SHAP feature: {top_feature}')

shap.dependence_plot(top_feature, sv, X_test, show=False)
plt.title(f'SHAP Dependence Plot — {top_feature}')
plt.tight_layout()
plt.show()

## 15. Comparative Interpretation and Analysis

In [ ]:
# Normalise each ranking to [0,1] for fair comparison
def normalise(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

shap_mean = pd.Series(np.abs(sv).mean(axis=0), index=X_test.columns)

compare = pd.DataFrame({
    'Mutual_Info'   : normalise(filter_df['Mutual_Info']),
    'RFE_Rank_inv'  : normalise(1 / rfe_df['Rank']),          # invert so higher = more important
    'Lasso_Coef'    : normalise(lasso_coef),
    'RF_MDI'        : normalise(rf_importance),
    'Permutation'   : normalise(perm_df['Importance_Mean']),
    'SHAP'          : normalise(shap_mean)
}).fillna(0).sort_values('SHAP', ascending=False)

print(compare.round(3))

compare.plot(kind='bar', figsize=(14, 6), colormap='tab10')
plt.title('Comparative Feature Importance Across All Methods (Normalised)')
plt.ylabel('Normalised Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 15.1 Correlation Heatmap of Rankings

In [ ]:
corr = compare.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Rank Correlation Between Feature Importance Methods')
plt.tight_layout()
plt.show()